# 01 - Run all parameter sweeps on one consistent grid

This notebook runs every sweep in the project with the **same engine, the same endpoint
definitions and the same grid rule**, and saves the result tables to `results/tables/`.
Notebooks 02-07 read these tables (04 and 06 add one small table each); notebooks 08-11 compute and save their own.

**Grid rule (identical for every parameter).** 21 evenly spaced values over the paper's
Supplementary Information randomized range (log-spaced for the two doses), plus the base value
where it does not already fall on the grid. Everything that is not on an axis stays at the SI base
value and at the Fig. 2C dose (a0 = 2.15, i = 10). So each of the four parameter-group grids contains the same base point.

**Runs**

| Table | Axes | Purpose |
|---|---|---|
| `grid_alpha_betamin_c.csv` | alpha x beta_min x c | resistance cost and private benefit |
| `grid_gamma_ha_a0.csv` | gamma x h_a x a0 | antibiotic killing and dose |
| `grid_i_hi_db.csv` | i x h_i x d_b | inhibitor dose and action |
| `grid_kappab_da_phimax_xi.csv.gz` | kappa_b x d_a x phi_max x xi | antibiotic degradation and nutrient recycling |
| `grid_criterion_a0_100.csv` | alpha x beta_min x c at a0 = 100 | check of the paper's selection criterion |
| `grid_dose_a0_i.csv` | a0 x i, for four strains along the directions of the paper's Fig. 3A | the dose plane (no group swept both doses together) |
| `sweep_one_at_a_time.csv` | each of the 13 parameters alone | decoupled, directly comparable effects |
| `sweep_global_random.csv` | all strain parameters at once (10,000 random strains) | paper-style global ranking |

Set `FORCE_RERUN = True` to recompute a table that already exists. A full run takes roughly
25 minutes on two cores; about 16 of them are the four-parameter grid (194,481 runs).

In [1]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))
TABLES = PROJECT_ROOT / "results" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 160)

from src.model.ma2024_model import DEFAULT_PARAMS, SI_INITIAL_CONDITIONS, BASE_A0, BASE_I
from src.analysis import sweep_engine as engine
from src.analysis.grid_runner import (axis, run_grid, run_one_at_a_time, run_global_random,
                                      RANGES, BASE_VALUES, LOG_AXES, N_PER_AXIS)

FORCE_RERUN = False
print("Base parameters:", DEFAULT_PARAMS)
print("Initial conditions:", SI_INITIAL_CONDITIONS, "| base dose: a0 =", BASE_A0, ", i =", BASE_I)
print(f"Horizon stages (end time, sampling step): {engine.HORIZON_STAGES}")
print(f"Steady state: per-capita rates < {engine.TOL_SS} for the last {engine.SS_HOLD_FRACTION:.0%} of the run")
print(f"Extinction floor: {engine.EXTINCTION_FLOOR:.2e} (one cell if saturation = {engine.CELLS_AT_SATURATION:.0e} cells)")
print(f"Drug-free reference S (viability yardstick): {engine.no_drug_reference():.4f}")

Base parameters: Params(alpha=0.95, beta_min=0.9, c=0.7, gamma=1.38, h_a=3.0, h_i=2.0, kappa_b=0.35, d_a=0.02, d_b=1.0, xi=0.8, phi_max=1.0)
Initial conditions: {'ns0': 0.2, 'nr0': 0.2, 's0': 4.0, 'b0': 0.0} | base dose: a0 = 2.15 , i = 10.0
Horizon stages (end time, sampling step): ((150.0, 0.5), (600.0, 1.0), (2400.0, 2.0))
Steady state: per-capita rates < 0.001 for the last 10% of the run
Extinction floor: 4.40e-09 (one cell if saturation = 1e+09 cells)
Drug-free reference S (viability yardstick): 2.3350


## Grid design: one rule for all 13 parameters

In [2]:
design = pd.DataFrame([{
    "parameter": name, "base": BASE_VALUES[name], "min": RANGES[name][0], "max": RANGES[name][1],
    "spacing": "log" if name in LOG_AXES else "linear", "n values": len(axis(name)),
    "base on the 21-point grid": len(axis(name)) == N_PER_AXIS,
    "range source": ("group choice (not randomized in SI)" if name == "d_a" else
                     "SI dose range" if name in ("a0", "i") else "SI randomized range"),
} for name in RANGES])
design

,parameter,base,min,max,spacing,n values,base on the 21-point grid,range source
0,alpha,0.95,0.75,1.0,linear,21,True,SI randomized range
1,beta_min,0.90,0.00,1.0,linear,21,True,SI randomized range
2,c,0.70,0.00,1.0,linear,21,True,SI randomized range
3,gamma,1.38,1.10,1.4,linear,22,False,SI randomized range
4,h_a,3.00,1.00,5.0,linear,21,True,SI randomized range
5,h_i,2.00,1.00,5.0,linear,21,True,SI randomized range
6,kappa_b,0.35,0.00,1.0,linear,21,True,SI randomized range
7,d_b,1.00,1.00,10.0,linear,21,True,SI randomized range
8,xi,0.80,0.00,1.0,linear,21,True,SI randomized range
9,phi_max,1.00,0.00,5.0,linear,21,True,SI randomized range


In [3]:
def run_or_load(filename, runner):
    """Run a sweep and save it, or load it if the table already exists."""
    path = TABLES / filename
    if path.exists() and not FORCE_RERUN:
        df = pd.read_csv(path)
        print(f"{filename}: loaded {len(df):,} rows (already computed)")
        return df
    t0 = time.time()
    df = runner()
    df.to_csv(path, index=False, float_format="%.8g")
    dt = time.time() - t0
    print(f"{filename}: {len(df):,} runs in {dt:.0f} s ({dt / len(df) * 1000:.1f} ms per run)")
    return df

GRIDS = {
    "grid_alpha_betamin_c.csv": ["alpha", "beta_min", "c"],
    "grid_gamma_ha_a0.csv": ["gamma", "h_a", "a0"],
    "grid_i_hi_db.csv": ["i", "h_i", "d_b"],
    "grid_kappab_da_phimax_xi.csv.gz": ["kappa_b", "d_a", "phi_max", "xi"],
}

## The four parameter-group grids

In [4]:
tables = {}
for filename, names in GRIDS.items():
    tables[filename] = run_or_load(filename, lambda names=names: run_grid({k: axis(k) for k in names}))

grid_alpha_betamin_c.csv: loaded 9,261 rows (already computed)
grid_gamma_ha_a0.csv: loaded 10,164 rows (already computed)
grid_i_hi_db.csv: loaded 9,261 rows (already computed)


grid_kappab_da_phimax_xi.csv.gz: loaded 194,481 rows (already computed)


## Criterion grid, one-at-a-time sweeps and the global random sweep

- **Criterion grid:** the alpha x beta_min x c grid again at the top of the paper's dose range
  (a0 = 100, i = 10), the closest the dose range gets to the saturating limit that the paper's
  simplified criterion (Eq. 2) assumes.
- **Dose plane:** antibiotic dose x inhibitor dose for four strains along the directions of the paper's
  Fig. 3A (base; higher burden α = 0.8; higher private benefit β_min = 0.7, the reverse of the paper's
  panel; higher inhibition c = 0.9; the values are the group's choices).
- **One at a time:** each parameter varied alone (41 values, plus the base value for γ and a0) with everything else at base.
- **Global random:** 10,000 strains with all 11 strain parameters drawn at once (Latin hypercube; the paper's 10 randomized parameters plus d_a over the group's range),
  at the base dose. This mirrors the paper's randomized strains and ranks all parameters together.

In [5]:
tables["grid_criterion_a0_100.csv"] = run_or_load(
    "grid_criterion_a0_100.csv",
    lambda: run_grid({k: axis(k) for k in ["alpha", "beta_min", "c"]}, a0=100.0, i_dose=10.0))
STRAINS = {"base": {}, "higher burden (alpha = 0.8)": {"alpha": 0.8},
           "higher private benefit (beta_min = 0.7)": {"beta_min": 0.7},
           "higher inhibition (c = 0.9)": {"c": 0.9}}

def dose_plane():
    parts = []
    for name, over in STRAINS.items():
        part = run_grid({"a0": axis("a0"), "i": axis("i"), **{k: [v] for k, v in over.items()}})
        part = part.drop(columns=list(over))
        part.insert(0, "strain", name)
        parts.append(part)
    return pd.concat(parts, ignore_index=True)

tables["grid_dose_a0_i.csv"] = run_or_load("grid_dose_a0_i.csv", dose_plane)
tables["sweep_one_at_a_time.csv"] = run_or_load("sweep_one_at_a_time.csv", lambda: run_one_at_a_time(n=41))
tables["sweep_global_random.csv"] = run_or_load("sweep_global_random.csv", lambda: run_global_random(10000, seed=574))

grid_criterion_a0_100.csv: loaded 9,261 rows (already computed)
grid_dose_a0_i.csv: loaded 1,848 rows (already computed)
sweep_one_at_a_time.csv: loaded 535 rows (already computed)
sweep_global_random.csv: loaded 10,000 rows (already computed)


## Consistency checks

1. Every table has the same endpoint columns.
2. The base point gives the same result in every grid that contains it.
3. How many runs settled, collapsed or did not converge in each table.

In [6]:
endpoint_cols = None
for name, df in tables.items():
    cols = [c for c in df.columns if c not in RANGES and c not in ("parameter", "value", "strain")]
    endpoint_cols = cols if endpoint_cols is None else endpoint_cols
    assert cols == endpoint_cols, f"{name} has different endpoint columns"
print(f"All {len(tables)} tables share the same {len(endpoint_cols)} endpoint columns:")
print(endpoint_cols)

All 8 tables share the same 32 endpoint columns:
['ns_final', 'nr_final', 'total_final', 'f_R_final', 'diversity_final', 'diversity_min', 'ns_min', 'nr_min', 'total_min', 'peak_loss_S', 'peak_loss_R', 'peak_loss_total', 't_S_min', 't_regain_S', 't_90_final_S', 't_steady_state', 'ss_converged', 't_end', 't_a_below_1', 'delta_log_ratio', 'sensitive_survives', 'resistant_eliminated_abs', 'outcome', 'S_extinct_floor', 'R_extinct_floor', 't_extinct_S', 't_extinct_R', 'ns_final_floor', 'nr_final_floor', 'f_R_final_floor', 'outcome_floor', 'goal_met_floor']


In [7]:
check = ["f_R_final", "ns_final", "nr_final", "peak_loss_S", "t_steady_state", "t_regain_S", "diversity_final"]
rows = []
for filename, names in GRIDS.items():
    df = tables[filename]
    mask = np.ones(len(df), bool)
    for k in names:
        mask &= np.isclose(df[k], BASE_VALUES[k])
    assert mask.sum() == 1, f"{filename}: base point not found exactly once"
    rows.append({"table": filename, **df.loc[mask, check].iloc[0].to_dict()})
oat = tables["sweep_one_at_a_time.csv"]
base_oat = oat[(oat.parameter == "alpha") & np.isclose(oat.value, BASE_VALUES["alpha"])]
rows.append({"table": "sweep_one_at_a_time.csv", **base_oat[check].iloc[0].to_dict()})
dose = tables["grid_dose_a0_i.csv"]
base_dose = dose[(dose.strain == "base") & np.isclose(dose.a0, BASE_VALUES["a0"]) & np.isclose(dose.i, BASE_VALUES["i"])]
assert len(base_dose) == 1, "dose plane: base point not found exactly once"
rows.append({"table": "grid_dose_a0_i.csv", **base_dose[check].iloc[0].to_dict()})
base_table = pd.DataFrame(rows).set_index("table")
assert np.allclose(base_table.values, base_table.values[0], rtol=1e-6), "base point differs between tables"
print("Base point is identical in the six tables that contain it (the criterion grid is at a0 = 100; the random strains do not include it):")
base_table

Base point is identical in the six tables that contain it (the criterion grid is at a0 = 100; the random strains do not include it):


,f_R_final,ns_final,nr_final,peak_loss_S,t_steady_state,t_regain_S,diversity_final
table,,,,,,,
grid_alpha_betamin_c.csv,0.432555,2.174618,1.65768,0.394419,15.5,8.5,0.684022
grid_gamma_ha_a0.csv,0.432555,2.174618,1.65768,0.394419,15.5,8.5,0.684022
grid_i_hi_db.csv,0.432555,2.174618,1.65768,0.394419,15.5,8.5,0.684022
grid_kappab_da_phimax_xi.csv.gz,0.432555,2.174618,1.65768,0.394419,15.5,8.5,0.684022
sweep_one_at_a_time.csv,0.432555,2.174618,1.65768,0.394419,15.5,8.5,0.684022
grid_dose_a0_i.csv,0.432555,2.174618,1.65768,0.394419,15.5,8.5,0.684022


In [8]:
summary = []
for name, df in tables.items():
    summary.append({
        "table": name, "runs": len(df),
        "converged": int(df.ss_converged.sum()),
        "not converged": int((df.outcome == "not converged").sum()),
        "community collapsed": int((df.outcome == "community collapsed").sum()),
        "needed t > 150": int((df.t_end > 150).sum()),
        "crossed extinction floor": int((df.S_extinct_floor | df.R_extinct_floor).sum()),
    })
pd.DataFrame(summary).set_index("table")

,runs,converged,not converged,community collapsed,needed t > 150,crossed extinction floor
table,,,,,,
grid_alpha_betamin_c.csv,9261,9261,0,0,0,0
grid_gamma_ha_a0.csv,10164,10164,0,0,3568,2647
grid_i_hi_db.csv,9261,9261,0,0,0,0
grid_kappab_da_phimax_xi.csv.gz,194481,194128,3,350,462,353
grid_criterion_a0_100.csv,9261,9261,0,0,5945,6134
grid_dose_a0_i.csv,1848,1848,0,0,474,531
sweep_one_at_a_time.csv,535,535,0,0,20,21
sweep_global_random.csv,10000,9998,0,2,19,15


**Reading the last table.** "Community collapsed" runs are ones where the antibiotic has not
fallen below the killing threshold by the last horizon (t = 2,400), so both populations are still
decaying; with d_a = 0 and too little enzyme it never does. Their composition endpoints are stored as NaN so
they cannot leak into averages. "Crossed extinction floor" counts runs where a population fell
below one cell at some point; for those the `_floor` columns hold the result with that population
removed for good.